In [27]:
# refreshes every 24 hours
RIOT_API_KEY = "RGAPI-e211679b-1183-4b5d-b649-b995e9b28d87"

from huggingface_hub import login
import numpy as np
import pandas as pd 
import requests
from datasets import load_dataset
import os

In [22]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'region_americas': 'matches/region_americas-00000.parquet', 'region_asia': 'matches/region_asia-00000.parquet', 'region_europe': 'matches/region_europe-00000.parquet'}
df = pd.read_parquet("hf://datasets/gptilt/lol-basic-matches-challenger-10k/" + splits["region_americas"])

In [25]:
df

,endOfGameResult,gameDuration,gameEndTimestamp,gameId,gameStartTimestamp,gameVersion,matchId,platformId,region,team_100_EPIC_MONSTER_KILL,...,team_200_horde_first,team_200_horde_kills,team_200_inhibitor_first,team_200_inhibitor_kills,team_200_riftHerald_first,team_200_riftHerald_kills,team_200_tower_first,team_200_tower_kills,tournamentCode,winner_team_id
0,GameComplete,1400,1745018960446,5269865992,1745017559681,15.8.675.10,NA1_5269865992,NA1,americas,3,...,False,0,False,0,False,0,False,2,,100
1,GameComplete,1524,1744589457829,5266494442,1744587933314,15.7.672.4034,NA1_5266494442,NA1,americas,2,...,True,5,True,1,False,0,False,8,,200
2,GameComplete,1782,1744585171830,5266444763,1744583389986,15.7.672.4034,NA1_5266444763,NA1,americas,2,...,True,1,False,0,False,0,False,2,,100
3,GameComplete,1939,1744582474913,5266408515,1744580535842,15.7.672.4034,NA1_5266408515,NA1,americas,1001,...,False,3,True,1,False,0,False,8,,200
4,GameComplete,930,1744576222010,5266355238,1744575291787,15.7.672.4034,NA1_5266355238,NA1,americas,1,...,False,0,False,0,False,0,True,3,,200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3329,GameComplete,1306,1744695473229,1612786876,1744694166406,15.7.672.4034,LA1_1612786876,LA1,americas,1001,...,True,6,False,0,True,1,True,5,,200
3330,GameComplete,1517,1744690271585,1612758933,1744688754314,15.7.672.4034,LA1_1612758933,LA1,americas,1,...,True,3,True,2,False,0,False,7,,200
3331,GameComplete,1837,1744872208317,1613288313,1744870371493,15.8.673.8244,LA1_1613288313,LA1,americas,2,...,True,3,False,0,True,1,True,5,,100
3332,GameComplete,1902,1744776848623,1613000432,1744774946027,15.7.672.4034,LA1_1613000432,LA1,americas,3,...,False,0,True,2,False,0,False,10,,200


In [24]:
df["matchId"][:10]

0    NA1_5269865992
1    NA1_5266494442
2    NA1_5266444763
3    NA1_5266408515
4    NA1_5266355238
5    NA1_5266372395
6    NA1_5266320329
7    NA1_5266298635
8    NA1_5265904325
9    NA1_5267244886
Name: matchId, dtype: object

In [29]:
import pandas as pd
import requests
import time
from tqdm import tqdm

match_ids = df['matchId'].tolist()



def get_match_details(match_id, region='americas'):
    """Fetch match details from Riot API"""
    url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    headers = {"X-Riot-Token": RIOT_API_KEY}
    
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        return response.json()
    elif response.status_code == 429:  # Rate limit
        time.sleep(2)
        return get_match_details(match_id, region)
    else:
        print(f"Error {response.status_code} for match {match_id}")
        return None

def extract_player_features(match_data):
    """Extract all white-label features for each player"""
    players = []
    
    info = match_data['info']
    match_id = match_data['metadata']['matchId']
    
    for participant in info['participants']:
        player = {
            'matchId': match_id,
            'puuid': participant['puuid'],
            
            # Performance stats
            'kills': participant['kills'],
            'deaths': participant['deaths'],
            'assists': participant['assists'],
            'kda': (participant['kills'] + participant['assists']) / max(participant['deaths'], 1),
            
            # Combat metrics
            'totalDamageDealtToChampions': participant['totalDamageDealtToChampions'],
            'totalDamageTaken': participant['totalDamageTaken'],
            
            # Economy
            'goldEarned': participant['goldEarned'],
            'totalMinionsKilled': participant['totalMinionsKilled'],
            'neutralMinionsKilled': participant['neutralMinionsKilled'],
            
            # Vision control
            'visionScore': participant['visionScore'],
            'wardsPlaced': participant['wardsPlaced'],
            'wardsKilled': participant['wardsKilled'],
            
            # Objectives
            'turretKills': participant['turretKills'],
            'inhibitorKills': participant['inhibitorKills'],
            'firstBloodKill': participant['firstBloodKill'],
            
            # Team contribution
            'killParticipation': participant.get('challenges', {}).get('killParticipation', 0),
            'teamDamagePercentage': participant.get('challenges', {}).get('teamDamagePercentage', 0),
            
            # Champion info
            'championName': participant['championName'],
            'championId': participant['championId'],
            
            # Game outcome
            'win': participant['win'],
            'gameDuration': info['gameDuration']
        }
        
        # Calculate per-minute stats
        duration_min = info['gameDuration'] / 60
        player['csPerMinute'] = (player['totalMinionsKilled'] + player['neutralMinionsKilled']) / duration_min
        player['goldPerMinute'] = player['goldEarned'] / duration_min
        player['visionScorePerMinute'] = player['visionScore'] / duration_min
        
        players.append(player)
    
    return players

# Collect data
all_players = []

for match_id in tqdm(match_ids):  # Start with 100 matches for testing
    match_data = get_match_details(match_id)
    
    if match_data:
        players = extract_player_features(match_data)
        all_players.extend(players)
    
    time.sleep(1.2)  # Rate limit: 20 requests per second, be safe with 1.2s

# Create DataFrame
df_players = pd.DataFrame(all_players)
df_players.to_csv("lol_player_features.csv", index=False)

print(f"\nCollected data for {len(df_players)} players across {len(df_players['matchId'].unique())} matches")
print(f"\nColumns: {df_players.columns.tolist()}")
print(f"\nSample:")
print(df_players.head())

100%|██████████| 3334/3334 [1:30:36<00:00,  1.63s/it]  
